# YorubaLlama -- Zero-Shot Quality Check (before committing to a full fine-tune)

**Purpose:** `Jacaranda/YorubaLlama` (Llama-3-8B, continually pretrained on 8.1GB of Yoruba text + instruction-tuned on 66,280 Yoruba pairs) looks like the best-documented Yoruba-capable candidate found so far -- its own model card shows a real, coherent, on-topic answer to a health question. But that example is the developer's own promotional pick, not independently verified. This notebook tests it on OUR actual demo questions, using OUR actual prompt template, before spending any GPU time on a full fine-tune.

**This is NOT a training run** -- just loading the base model (no LoRA, no fine-tuning) and generating real answers to see how it does out of the box.

## Before you start
1. Visit https://huggingface.co/Jacaranda/YorubaLlama while logged in and click **"Agree and access repository"** (usually instant) -- skip if you already did this.
2. Get a token at https://huggingface.co/settings/tokens (Read-only is enough).
3. In this notebook: **Add-ons -> Secrets -> Add secret**, label `HF_TOKEN`, value = your token, and make sure it's attached to this notebook.
4. In the right-hand sidebar: turn on a **GPU** accelerator and switch **Internet** to **On**.
5. Run all cells (Run -> Run All), or **Save Version -> Save & Run All** if you'd rather it run in the background.

In [ ]:
# Confirm a GPU is attached. This is an 8B model -- needs a GPU (T4 or better) to load in a reasonable time even in 4-bit.
!nvidia-smi

In [ ]:
!pip install -q -U transformers peft bitsandbytes accelerate

In [ ]:
# Get the pipeline CODE (not data) so we can import the REAL prompt
# template (src/model/prompt.py) directly, instead of retyping it --
# guarantees byte-for-byte fidelity with what the actual app/training
# pipeline uses, no risk of a manual transcription error.
import shutil

%cd /kaggle/working
shutil.rmtree("Yoruba-HealthQA", ignore_errors=True)
!git clone https://github.com/ayoolaeni/Yoruba-HealthQA.git
%cd Yoruba-HealthQA

In [ ]:
# Log in via Kaggle Secrets (see step 3 above) -- falls back to a typed prompt
# if run interactively without a secret set.
from huggingface_hub import login

try:
    from kaggle_secrets import UserSecretsClient
    _token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    import getpass
    _token = getpass.getpass("Paste your Hugging Face access token: ")
login(token=_token)

In [ ]:
# Load the base model in 4-bit (matches how our real pipeline would use it via QLoRA later -- same memory profile, no fine-tuning happening here).
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE_MODEL = "Jacaranda/YorubaLlama"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb_config, device_map="auto")
model.eval()
print("Loaded.")

In [ ]:
# SAFETY CHECK: confirm this model's real layer names match what our LoRA
# training config assumes (q_proj/k_proj/v_proj/o_proj/gate_proj/up_proj/down_proj)
# -- this is a standard Llama-3 architecture so it should match, but verify
# rather than assume, same discipline as every other model checked so far.
EXPECTED_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
linear_suffixes = sorted({name.rsplit(".", 1)[-1] for name, mod in model.named_modules()
                           if mod.__class__.__name__ in ("Linear", "Linear4bit") and "." in name})
print("Real Linear-layer name suffixes found:", linear_suffixes)
missing = [m for m in EXPECTED_TARGET_MODULES if m not in linear_suffixes]
assert not missing, f"Mismatch: expected {EXPECTED_TARGET_MODULES} but missing {missing}. Update before fine-tuning."
print("OK: matches the standard Llama target_modules our pipeline already uses.")

In [ ]:
# Real zero-shot test on OUR actual demo questions, using OUR actual prompt
# template -- imported DIRECTLY from src/model/prompt.py (the cloned repo),
# not retyped, so there is zero risk of a transcription error in the
# Yoruba system prompt. Not the model card's own cherry-picked example
# either -- this is exactly how the app would call it. Two versions
# tested per question: our full system-prompted format, and a bare
# question (closer to how the model card's own example was likely run),
# so we can see whether our template helps, hurts, or makes no difference.
import time

from src.model.prompt import build_prompt

questions = [
    "Kí ni àwọn àmì àrùn ibà?",              # in-scope: malaria symptoms
    "Kí ló máa ń fa ibà ní ara ènìyàn?",       # the earlier real bug case: what causes fever?
    "Kí ni àwọn àmì àrùn àtọ̀gbẹ?",           # diabetes symptoms
]

def generate(prompt, max_new_tokens=120):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, repetition_penalty=1.3)
    text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return text.strip(), time.time() - t0

for q in questions:
    print("=" * 70)
    print("Q:", q)
    text, dt = generate(build_prompt(q))  # our real prompt template, straight from source
    print(f"[OUR PROMPT TEMPLATE, {dt:.1f}s] {text}")
    text2, dt2 = generate(q)  # bare question, no template
    print(f"[BARE QUESTION, {dt2:.1f}s] {text2}")
    print()

## What to look at

For each question, compare the two outputs:
- Is it real, coherent Yoruba (not invented words)?
- Does it actually answer the question asked (not a different, unrelated topic)?
- Is it medically sensible, even if not perfectly complete?
- How long did it take (relevant for whether live generation is even feasible later, though our retrieval demo doesn't need this at all)?

Copy the full output here (or download this notebook's log) so it can be reviewed properly before deciding whether to invest GPU time in a full fine-tune on top of this base model.